In [192]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from keras.layers import Dense
from keras.models import Sequential
from keras.callbacks import EarlyStopping,TensorBoard

import datetime

import pickle

In [193]:
df = pd.read_csv("Churn_Modelling.csv")

In [194]:
df.head()

,CustomerId,Name,HasCrCard,CreditScore,Geography,Gender,Age,Tenure,Balance,PrdtcCnts,ActivedMmbr,EstimatedSalary,Exited
0,15634602,Hargrave,1,619,France,Female,42,2,0.00,1,1,101348.88,1
1,15647311,Hill,0,608,Spain,Female,41,1,83807.86,1,1,112542.58,0
2,15619304,Onio,1,502,France,Female,42,8,159660.80,3,0,113931.57,1
3,15701354,Boni,0,699,France,Female,39,1,0.00,2,0,93826.63,0
4,15737888,Mitchell,1,850,Spain,Female,43,2,125510.82,1,1,79084.10,0


In [195]:
df = df.drop(['CustomerId', 'Name'], axis = 1)

In [196]:
df.head()

,HasCrCard,CreditScore,Geography,Gender,Age,Tenure,Balance,PrdtcCnts,ActivedMmbr,EstimatedSalary,Exited
0,1,619,France,Female,42,2,0.00,1,1,101348.88,1
1,0,608,Spain,Female,41,1,83807.86,1,1,112542.58,0
2,1,502,France,Female,42,8,159660.80,3,0,113931.57,1
3,0,699,France,Female,39,1,0.00,2,0,93826.63,0
4,1,850,Spain,Female,43,2,125510.82,1,1,79084.10,0


In [197]:
df.isnull().sum()

HasCrCard          0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
PrdtcCnts          0
ActivedMmbr        0
EstimatedSalary    0
Exited             0
dtype: int64

In [198]:
le = LabelEncoder()
df["Gender"] = le.fit_transform(df["Gender"])

In [199]:
ohe = OneHotEncoder()
ohe_geo = ohe.fit_transform(df[['Geography']])

In [200]:
ohe_geo

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [201]:
ohe_geo.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [202]:
ohe.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [203]:
df_geo = pd.DataFrame(ohe_geo.toarray(), columns = ohe.get_feature_names_out(['Geography']))

In [204]:
df_geo.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [205]:
df = pd.concat([df.drop('Geography', axis=1), df_geo], axis=1)
# df = df.drop('Geography', axis = 1)
# df = pd.concat([df, df_geo], axis=1)

In [206]:
df

,HasCrCard,CreditScore,Gender,Age,Tenure,Balance,PrdtcCnts,ActivedMmbr,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,1,619,0,42,2,0.00,1,1,101348.88,1,1.0,0.0,0.0
1,0,608,0,41,1,83807.86,1,1,112542.58,0,0.0,0.0,1.0
2,1,502,0,42,8,159660.80,3,0,113931.57,1,1.0,0.0,0.0
3,0,699,0,39,1,0.00,2,0,93826.63,0,1.0,0.0,0.0
4,1,850,0,43,2,125510.82,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,1,771,1,39,5,0.00,2,0,96270.64,0,1.0,0.0,0.0
9996,1,516,1,35,10,57369.61,1,1,101699.77,0,1.0,0.0,0.0
9997,0,709,0,36,7,0.00,1,1,42085.58,1,1.0,0.0,0.0
9998,1,772,1,42,3,75075.31,2,0,92888.52,1,0.0,1.0,0.0


In [207]:
with open('le.pkl','wb') as file:
    pickle.dump(le, file)

with open('ohe.pkl','wb') as file:
    pickle.dump(ohe, file)

In [208]:
X = df.drop('Exited', axis = 1)
Y = df['Exited']

In [209]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size = 0.2, random_state =42)

In [210]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [211]:
with open('scaler', 'wb') as file:
    pickle.dump(scaler, file)

In [212]:
model = Sequential([
    Dense(64,activation = 'relu', input_shape = (X_train.shape[1],)), # First Hidden Layer and connected with input layer
    Dense(32, activation = 'relu'), # Hidden Layer 2
    Dense(1, activation = 'sigmoid') # Output Layer
])


C:\Users\Lalit Chandra\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [213]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                      │ (None, 64)                  │             832 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_9 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [214]:
# Setup TensorBoard
log_dir = "logs/fit" + datetime.datetime.now().strftime("%d%m%Y-%H%M%S")
tensorflow_calllback= TensorBoard(log_dir=log_dir,histogram_freq=1)

In [215]:
model.compile(optimizer='adam', loss ='binary_crossentropy', metrics=['accuracy'])

In [216]:
# Setup Early Stopping
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [217]:
results = model.fit(X_train,Y_train, validation_data=(X_test, Y_test), epochs =100,
                   callbacks=[tensorflow_calllback, early_stopping_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8001 - loss: 0.4515 - val_accuracy: 0.8300 - val_loss: 0.3916
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8429 - loss: 0.3802 - val_accuracy: 0.8545 - val_loss: 0.3574
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8535 - loss: 0.3558 - val_accuracy: 0.8530 - val_loss: 0.3496
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8626 - loss: 0.3464 - val_accuracy: 0.8585 - val_loss: 0.3501
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8604 - loss: 0.3432 - val_accuracy: 0.8590 - val_loss: 0.3472
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8616 - loss: 0.3401 - val_accuracy: 0.8600 - val_loss: 0.3429
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8608 - loss: 0.3370 - val_accuracy: 0.8590 - val_loss: 0.3446
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8644 - loss: 0.3334 - val_accu

In [218]:
model.save('model.h5')